# Graph-Med: Exploring Clinical Knowledge with Ontologies, Agents, and Graph Neural Networks

This notebook demonstrates the full pipeline:
1. **Connectors** — Embedding API, LLM, Neo4j, GNN Reranker, GRetriever
2. **Knowledge Graph** — HPO/ICD ontology structure and traversal
3. **Ontology Mapping** — Embedding retrieval + LLM disambiguation
4. **GNN-Enhanced Mapping** — Structural reranking with GAT
5. **GRetriever** — Joint GNN+LLM with PCST subgraphs and LoRA
6. **Patient Annotation** — NER and NED on clinical text
7. **GraphRAG Agent** — LangGraph agent with pluggable mapping backends
8. **Three-Way Comparison** — Real patient P003, 68 ICD codes, 3 methods

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Imports ---
import json
import asyncio
import pandas as pd
from openai import OpenAI, AsyncOpenAI
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph
from IPython.display import JSON, Markdown, display

from util.config_loader import load_config_api
from util.api_client import ApiClient
from llm.tool import (
    build_ontology_mapper_tool,
    build_patient_ner_tool,
    build_patient_ned_tool,
    build_general_medical_tool,
    build_patient_info_tool,
    build_patient_coverage_tool,
)


def show_json(raw, mode="text", indent=2):
    """Display JSON output as formatted markdown."""
    if isinstance(raw, str):
        raw = json.loads(raw)
    if mode in ("tree", "both"):
        display(JSON(raw))
    if mode in ("text", "both"):
        display(Markdown(f"```json\n{json.dumps(raw, indent=indent, ensure_ascii=False)}\n```"))

## 2. Connectors

The platform uses 5 services:

| Service | Technology | Purpose |
|---------|-----------|--------|
| **Embedding API** | GTE-Qwen2-7B (Colab) | Vector embeddings for semantic search |
| **LLM** | MedGemma-4b-it (Colab) | Medical text generation and disambiguation |
| **Neo4j** | Neo4j Desktop (local) | Knowledge graph storage and Cypher queries |
| **GNN Reranker** | ResidualGNN/GAT (Colab) | Structural reranking of HPO candidates |
| **GRetriever** | GNN+MLP+LoRA MedGemma (Colab) | Graph-based LLM disambiguation |

### Embedding API (GTE-Qwen2-7B)

In [3]:
# Connect to the embedding server and verify with a semantic similarity example
url_emb = load_config_api("embedding", path="../config.ini")
api = ApiClient(url_emb)

# Embed three medical terms: two synonyms + one related but distinct
terms = ["Shortness of breath", "Dyspnea", "Tachypnea"]
body, status, _ = api.post('/v1/embeddings', {'input': terms})
print(f"Embedding API: status={status}, model={body['model']}")
print(f"  Embedding dim: {len(body['data'][0]['embedding'])}")

import numpy as np
embs = [np.array(body['data'][i]['embedding']) for i in range(3)]

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\n  Cosine similarities:")
print(f"    '{terms[0]}' <-> '{terms[1]}' = {cosine(embs[0], embs[1]):.4f}  (synonyms)")
print(f"    '{terms[0]}' <-> '{terms[2]}' = {cosine(embs[0], embs[2]):.4f}  (related but distinct)")
print(f"    '{terms[1]}' <-> '{terms[2]}' = {cosine(embs[1], embs[2]):.4f}  (related but distinct)")
print(f"\n  Embeddings capture that Dyspnea = Shortness of breath (subjective sensation),")
print(f"  while Tachypnea (rapid breathing rate) is related but clinically different.")

Embedding API: status=200, model=Alibaba-NLP/gte-Qwen2-7B-instruct
  Embedding dim: 3584

  Cosine similarities:
    'Shortness of breath' <-> 'Dyspnea' = 0.7982  (synonyms)
    'Shortness of breath' <-> 'Tachypnea' = 0.7054  (related but distinct)
    'Dyspnea' <-> 'Tachypnea' = 0.7579  (related but distinct)

  Embeddings capture that Dyspnea = Shortness of breath (subjective sensation),
  while Tachypnea (rapid breathing rate) is related but clinically different.


### LLM (MedGemma-4b-it)

In [4]:
# LangChain wrapper for MedGemma chat completions
url_llm = load_config_api("llm", path="../config.ini")
chat_client = ChatOpenAI(
    api_key="EMPTY",
    base_url=url_llm,
    model_name="google/medgemma-4b-it",
    temperature=0,
    max_tokens=8192,
    top_p=0.9,
    frequency_penalty=0.2,
    presence_penalty=0.0,
)

# Same clinical distinction — now explained by the LLM
client = OpenAI(api_key="EMPTY", base_url=url_llm)
resp = client.chat.completions.create(
    model="google/medgemma-4b-it",
    messages=[{"role": "user", "content": (
        "Who are you? And what is the clinical difference between Shortness of breath (Dyspnea) "
        "and Tachypnea? Answer in 2-3 sentences."
    )}],
    temperature=0,
)
print(f"LLM: {resp.choices[0].message.content}")

LLM: I am a large language model, trained by Google.

The clinical difference between dyspnea and tachypnea is that dyspnea is the subjective sensation of difficulty breathing, while tachypnea is the objective measurement of an increased respiratory rate. Dyspnea is a symptom, while tachypnea is a sign.



### Neo4j Knowledge Graph

In [5]:
# Connect to Neo4j and verify with a count query
graph_client = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="password",
    database="nodes2026",
    enhanced_schema=True,
)

counts = graph_client.query("""
    MATCH (n)
    WITH labels(n) AS lbls
    UNWIND lbls AS lbl
    RETURN lbl AS label, count(*) AS count
    ORDER BY count DESC
    LIMIT 8
""")
print("Neo4j node counts:")
for r in counts:
    print(f"  {r['label']:<20} {r['count']:>6}")

Neo4j node counts:
  Umls                  48724
  Resource              32940
  HpoPhenotype          19944
  Class                 19944
  HpoDisease            12996
  IcdDisease            12221
  ProcessedWithOntologyMapper  12150
  IcdGroup                 22


### GNN Reranker API (ResidualGNN)

In [6]:
# Connect to the GNN reranking server and test with a single query
url_gnn = load_config_api("gnn", path="../config.ini")
gnn_api = ApiClient(url_gnn)

body, status, _ = gnn_api.get('/healthz')
print(f"GNN server: status={status}")
print(f"  Model: {body['model']}, alpha={body['alpha']}")
print(f"  Nodes: {body['nodes']}, Edges: {body['edges']}")

# Quick compare: M54.2 Cervicalgia — GNN rescues "Neck pain" from Qwen rank 5 to #1
result, _, _ = gnn_api.post('/compare', {
    'icd_code': 'M54.2',
    'candidate_codes': [
        'HP:0008480', 'HP:6000107', 'HP:0030009', 'HP:0032535', 'HP:0030833',
    ],
    'top_k': 5,
})
print(f"\n  Compare example: ICD M54.2 ({result['icd_label']})")
print(f"  {'Qwen':>6} {'GNN':>6} {'Delta':>6}  {'HPO Code':<12} {'Label'}")
print(f"  {'-'*55}")
for c in result['by_gnn']:
    delta_str = f"+{c['rank_delta']}" if c['rank_delta'] > 0 else str(c['rank_delta'])
    print(f"  Q {c['qwen_rank']:>2}  G {c['gnn_rank']:>2}  {delta_str:>5}   {c['code']:<12} {c['label']}")

GNN server: status=200
  Model: ResidualGNN, alpha=0.102
  Nodes: 45205, Edges: 330488

  Compare example: ICD M54.2 (Cervicalgia)
    Qwen    GNN  Delta  HPO Code     Label
  -------------------------------------------------------
  Q  5  G  1     +4   HP:0030833   Neck pain
  Q  1  G  2     -1   HP:0008480   Cervical spondylosis
  Q  3  G  3      0   HP:0030009   Cervical insufficiency
  Q  2  G  4     -2   HP:6000107   Cervical motion tenderness
  Q  4  G  5     -1   HP:0032535   Cervical (neck)


**Alpha (α)** is the learned residual scale: `refined = original_qwen_embedding + α × correction`,
where *correction* is the GAT's 2-hop message-passing output projected back to 3584 dims.
A small α (≈0.1) means the GNN only slightly nudges the Qwen embedding based on
structural context — enough to reorder close candidates without destroying the semantic baseline.

### GRetriever API (GNN + LoRA MedGemma)

In [7]:
# Connect to the GRetriever server
url_gret = load_config_api("gretriever", path="../config.ini")
gret_api = ApiClient(url_gret)

body, status, _ = gret_api.get('/healthz')
print(f"GRetriever server: status={status}")
print(f"  Model: {body['model']}, LLM: {body['llm']}")
print(f"  Max key nodes: {body['max_key_nodes']}")
print(f"  Nodes: {body['nodes']}, Edges: {body['edges']}")

# Quick test: R06.3 Periodic breathing
result, _, _ = gret_api.post('/compare', {
    'icd_code': 'R06.3',
    'candidate_codes': [
        'HP:0004879', 'HP:0012196', 'HP:0012195', 'HP:0005957', 'HP:0002793',
        'HP:0005941', 'HP:0030207', 'HP:0004881', 'HP:0040213', 'HP:0002877',
    ],
})
print(f"\n  Test: ICD R06.3 ({result['icd_label']})")
print(f"    Text-only:   {result['text_only']['code']}  {result['text_only']['name']}")
print(f"    GRetriever:  {result['gretriever']['code']}  {result['gretriever']['name']}")


GRetriever server: status=200
  Model: GRetrieverPCST, LLM: medgemma-4b-it+LoRA
  Max key nodes: 16
  Nodes: 45205, Edges: 330488

  Test: ICD R06.3 (Periodic breathing)
    Text-only:   HP:0004879  Intermittent hyperventilation
    GRetriever:  HP:0012196  Cheyne-Stokes respiration


## 3. Knowledge Graph Overview

The graph contains two medical ontologies linked by UMLS:
- **ICD-10** — 12,221 diagnostic codes (diseases, chapters, groups)
- **HPO** — 19,944 phenotype terms + 12,996 disease definitions
- **UMLS** — cross-ontology bridge connecting ICD codes to HPO phenotypes

### HPO Disease-Phenotype Relationships

In [8]:
# Find diseases associated with a phenotype category
graph_client.query("""
    MATCH (p:HpoPhenotype {label: "Abnormality of the endocrine system"})
          <-[:HAS_PHENOTYPIC_FEATURE]-(d:HpoDisease)
    RETURN d.label AS disease
    LIMIT 10
""")

[{'disease': 'Biemond syndrome II'},
 {'disease': 'Pseudovaginal perineoscrotal hypospadias'},
 {'disease': 'Ectodermal dysplasia with adrenal cyst'},
 {'disease': 'Thymic-Renal-Anal-Lung dysplasia'},
 {'disease': 'Myasthenia gravis'},
 {'disease': 'THIOUREA TASTINGPHENYLTHIOCARBAMIDE TASTING, INCLUDED'},
 {'disease': 'MENOPAUSE, NATURAL, AGE AT, QUANTITATIVE TRAIT LOCUS 1'},
 {'disease': 'Candidiasis, familial chronic mucocutaneous, autosomal dominant'},
 {'disease': 'PYGMY'},
 {'disease': 'PURA-related severe neonatal hypotonia-seizures-encephalopathy syndrome'}]

### Ontology Hierarchy Traversal (rdfs:subClassOf)

In [9]:
# Traverse the HPO hierarchy: find diseases under a phenotype category
# and collect all their phenotypic features, highlighting which ones
# belong to the "Abnormality of the endocrine system" subtree
graph_client.query("""
    MATCH (cat:HpoPhenotype {label: "Abnormality of the endocrine system"})
    MATCH (sub)-[:subClassOf*0..]->(cat)
    WITH cat, collect(DISTINCT sub) AS endocrine_subtree
    MATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(sub) WHERE sub IN endocrine_subtree
    MATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
    WITH dis, endocrine_subtree,
         collect(DISTINCT phe.label) AS all_features,
         [p IN collect(DISTINCT phe) WHERE p IN endocrine_subtree | p.label] AS endocrine_features
    RETURN dis.label AS disease,
           endocrine_features AS endocrine_features,
           [f IN all_features WHERE NOT f IN endocrine_features] AS other_features,
           size(endocrine_features) AS n_endocrine,
           size(all_features) AS n_total
    ORDER BY n_total ASC, disease
    SKIP 100 LIMIT 5
""")

[{'disease': 'Adiponectin deficiency',
  'endocrine_features': ['Type II diabetes mellitus',
   'Decreased adiponectin level'],
  'other_features': ['Autosomal dominant inheritance',
   'Coronary artery atherosclerosis',
   'Adult onset',
   'Stage 5 chronic kidney disease'],
  'n_endocrine': 2,
  'n_total': 6},
 {'disease': 'Adrenal hypoplasia, congenital, with absent pituitary luteinizinghormone',
  'endocrine_features': ['Congenital adrenal hypoplasia',
   'Decreased circulating luteinizing hormone level'],
  'other_features': ['Autosomal recessive inheritance',
   'Cryptorchidism',
   'Micropenis',
   'Neonatal onset'],
  'n_endocrine': 2,
  'n_total': 6},
 {'disease': 'Adrenal insufficiency, congenital, with 46XY sex reversal, partial or complete',
  'endocrine_features': ['Adrenal insufficiency',
   'Increased circulating aldosterone concentration',
   'Adrenocorticotropic hormone excess'],
  'other_features': ['Renal salt wasting',
   'Hyperpigmentation of the skin',
   'Sex rev

### UMLS Bridge: ICD-10 to HPO Paths

UMLS concepts link ICD codes to HPO phenotypes via shared CUI identifiers.
Only ~2.2% of ICD codes have direct HPO mappings through UMLS.

In [10]:
# Extract ICD→UMLS→HPO paths
graph_client.query("""
    MATCH path = (d:IcdDisease)<-[:UMLS_TO_ICD]-(:Umls)-[:UMLS_TO_HPO_PHENOTYPE]->(p:HpoPhenotype)
    WITH d, p,
         [n IN nodes(path) | COALESCE(n.label, n.id, elementId(n))] AS node_names,
         [r IN relationships(path) | type(r)] AS rel_types,
         [r IN relationships(path) | COALESCE(startNode(r).label, startNode(r).id)] AS rel_starts
    WITH [i IN range(0, size(node_names) - 1) |
        CASE
            WHEN i = size(node_names) - 1
                THEN '(' + node_names[i] + ')'
            WHEN node_names[i] = rel_starts[i]
                THEN '(' + node_names[i] + ')' + '-[:' + rel_types[i] + ']->'
            ELSE '(' + node_names[i] + ')' + '<-[:' + rel_types[i] + ']-'
        END
    ] AS string_paths
    RETURN DISTINCT apoc.text.join(string_paths, '') AS `Extracted path`
    LIMIT 5
""")

[{'Extracted path': '(Paratyphoid fever, unspecified)<-[:UMLS_TO_ICD]-(C0015672)-[:UMLS_TO_HPO_PHENOTYPE]->(Fatigue)'},
 {'Extracted path': '(Other salmonella infections)<-[:UMLS_TO_ICD]-(C0085593)-[:UMLS_TO_HPO_PHENOTYPE]->(Chills)'},
 {'Extracted path': '(Shigellosis)<-[:UMLS_TO_ICD]-(C0015967)-[:UMLS_TO_HPO_PHENOTYPE]->(Fever)'},
 {'Extracted path': '(Enterocolitis due to Clostridium difficile)<-[:UMLS_TO_ICD]-(C0238106)-[:UMLS_TO_HPO_PHENOTYPE]->(Clostridium difficile colitis)'},
 {'Extracted path': '(Other bacterial foodborne intoxications, not elsewhere classified)<-[:UMLS_TO_ICD]-(C0231218)-[:UMLS_TO_HPO_PHENOTYPE]->(Malaise)'}]

### Virtualized Patient Access (APOC Data Virtualization)

Patient data lives in an external FHIR-like source and is accessed on-the-fly
via APOC Data Virtualization — no patient records are stored in the graph itself.


In [11]:
# Virtualized patient access via APOC Data Virtualization
rows = graph_client.query("""
    CALL apoc.dv.query('encounter', {patientId: 'P003'}) YIELD node AS v
    RETURN v
    LIMIT 1
""")

v = rows[0]['v']
props = v if isinstance(v, dict) and 'properties' not in v else v.get('properties', v)

print(f"Patient: {props.get('patientId')}")
print(f"Encounter: {props.get('EncounterID')}")
print(f"Condition: {props.get('Condition')}")
print(f"Chief complaint: {props.get('ChiefComplaint')}")
print(f"Medication statement: {props.get('MedicationStatement')}")
print(f"Course trend: {props.get('CourseTrend')}")
print(f"Narrative summary: {props.get('Narrative')}")


Patient: P003
Encounter: E001
Condition: Multiple mononeuropathy
Chief complaint: Leg numbness and electric-like pains
Medication statement: Gabapentin 100 mg PO BID
Course trend: Worsened
Narrative summary: The patient reported several weeks of burning tingling in the feet, spreading intermittently up the calves. He described brief electric-like shocks and a sense of weakness after short walks but denied back trauma or toxin exposure. Examination revealed asymmetrical patchy loss of light touch and vibration in the distal legs with mildly broad-based gait. Nerve conduction testing demonstrated multifocal demyelinating changes suggestive of multiple mononeuropathy. He was started on low-dose gabapentin, advised to avoid overexertion, and scheduled for neurology follow-up to monitor evolution of symptoms.


## 4. ICD-to-HPO Ontology Mapping

The core challenge: translating ICD-10 diagnostic codes to HPO phenotype terms.
The pipeline has two stages:
1. **Candidate selection** — vector similarity retrieves top-K HPO candidates
2. **LLM disambiguation** — MedGemma picks the best match from candidates

### 4.1 Embedding-Based Candidate Selection

GTE-Qwen2-7B embeddings (dim=3584) stored on HPO nodes. Vector index enables
cosine similarity search.

In [12]:
# Vector search: find HPO phenotypes similar to a clinical term
user_query = "Shortness of breath"
q_embed = api.post('/v1/embeddings', {'input': [user_query]})[0]['data'][0]['embedding']

result = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 10, "qe": q_embed})

for rec in result:
    print(f"{rec['score']:.3f}  {rec['label']}  (id={rec['id']})")

0.899  Dyspnea  (id=HP:0002094)
0.866  Respiratory distress  (id=HP:0002098)
0.853  Tachypnea  (id=HP:0002789)
0.824  Exertional dyspnea  (id=HP:0002875)
0.823  Breathing dysregulation  (id=HP:0005957)
0.819  Respiratory insufficiency  (id=HP:0002093)
0.815  Rest dyspnea  (id=HP:0033710)
0.809  Paroxysmal dyspnea  (id=HP:0012763)
0.804  Wheezing  (id=HP:0030828)
0.803  Abnormal pattern of respiration  (id=HP:0002793)


### 4.2 LLM Disambiguation

MedGemma selects the best HPO match from the candidate list, using ICD context
(parent, group, chapter) to resolve ambiguity.

In [13]:
# LLM-based disambiguation for R07.1 Chest pain on breathing
payload = {
    "source_concept": "R07.1 Chest pain on breathing",
    "source_context": """
    {
      "id": "R07.1",
      "name": "Chest pain on breathing",
      "parentName": "Pain in throat and chest",
      "group":   { "groupName": "Symptoms and signs involving the circulatory and respiratory systems" },
      "chapter": { "chapterName": "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified" }
    }
    """,
    "candidate_list": """
    [
      {
        "score": 0.93,
        "id": "HP:0002104",
        "label": "Chest pain",
        "exactSynonym": [
          "Thoracic pain"
        ],
        "description": "Pain localized to the anterior or posterior chest wall, without specification of the precipitating factor, temporal pattern, or relationship to respiration or exertion. This term is intentionally broad and may encompass musculoskeletal, cardiac, pulmonary, gastrointestinal, or idiopathic etiologies when the clinical context is not further specified."
      },
      {
        "score": 0.90,
        "id": "HP:0100749",
        "label": "Exertional chest pain",
        "exactSynonym": [
          "Chest pain on exertion"
        ],
        "description": "Chest discomfort or pain that is primarily triggered, precipitated, or worsened by physical activity or emotional stress and tends to improve with rest. This feature is classically associated with myocardial ischemia or other cardiopulmonary limitations related to increased workload, rather than with respiratory movements such as inspiration or coughing."
      },
      {
        "score": 0.87,
        "id": "HP:0030165",
        "label": "Pleuritic chest pain",
        "exactSynonym": [
          "Pleural pain"
        ],
        "description": "A sharp, stabbing, or burning chest pain that is characteristically exacerbated by deep inspiration, coughing, sneezing, or other movements of the chest wall and diaphragm, and often relieved by lying on the affected side. This symptom is typically associated with inflammation or irritation of the pleura (e.g., pleuritis, pulmonary embolism, or pneumonia) and corresponds clinically to chest pain that occurs specifically on breathing."
      }
    ]
    """,
}


tool_output = build_ontology_mapper_tool(chat_client).invoke(payload)
show_json(tool_output)


/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OntologyMappingResponse(b...oncept's description.")), input_type=OntologyMappingResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "best_id": "HP:0030165",
  "best_label": "Pleuritic chest pain",
  "confidence": 0.93,
  "rationale": "The source concept describes chest pain on breathing, which is a key feature of pleuritic chest pain. The description of pleuritic chest pain aligns with the source concept's clinical context.",
  "support": {
    "evidence": "The description of pleuritic chest pain includes 'chest pain on breathing'.",
    "reason": "Pleuritic chest pain is characterized by sharp, stabbing pain exacerbated by breathing, which directly corresponds to the source concept's description."
  }
}
```

### 4.3 Pre-Computed Mapping Results

Path A mappings are stored as `ICD_MAPS_TO_HPO_BY_EMBEDDING` relationships in Neo4j.

In [14]:
# Query a stored ICD→HPO mapping with confidence and rationale
result = graph_client.query("""
    MATCH (d:IcdDisease)-[r:ICD_MAPS_TO_HPO_BY_EMBEDDING]->(p:HpoPhenotype)
    RETURN d.id AS icd_id, d.label AS icd_label,
           p.id AS hpo_id, p.label AS hpo_label,
           r.confidence AS confidence, r.evidence AS evidence, r.rationale AS rationale
    OFFSET 100 LIMIT 1
""")
for rec in result:
    print(f"ICD: {rec['icd_id']} ({rec['icd_label']})")
    print(f"HPO: {rec['hpo_id']} ({rec['hpo_label']})")
    print(f"Confidence: {rec['confidence']}, Evidence: {rec['evidence']}")
    print(f"Rationale: {rec['rationale']}")

ICD: A18.4 (Tuberculosis of skin and subcutaneous tissue)
HPO: HP:0032271 (Extrapulmonary tuberculosis)
Confidence: 0.9, Evidence: None
Rationale: The source concept is 'Tuberculosis of skin and subcutaneous tissue', which is a type of extrapulmonary tuberculosis. The candidate 'Extrapulmonary tuberculosis' is the most specific and semantically equivalent concept.


## 5. GNN-Enhanced Ontology Mapping

### The Disambiguation Problem

Vector search retrieves semantically similar HPO candidates, but struggles when
multiple terms share lexical overlap. The ResidualGNN (2-layer GAT) enriches each
node's embedding with ontology hierarchy context via message passing.

### Aggregate Results

- **368 test ICD codes**, 34 Qwen failures (correct HPO ranked > 5)
- **17/34 (50%) rescued** to top-5 by GNN reranking
- Notable rescues: M54.2 (rank 19->1), Q76.6 (rank 11->1), C18.9 (rank 7->1)

### 5.1 Hard Case: M54.2 Cervicalgia -- Qwen Fails, GNN Rescues

Qwen buries "Neck pain" (the correct HPO) at rank 19, favoring "Cervical spondylosis"
and other terms that share the word "cervical". The GNN pushes "Neck pain" to rank 1.

In [15]:
# Hard case: M54.2 Cervicalgia
icd_code = "M54.2"

# Step 1: Qwen vector search for top-20 HPO candidates
icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
    {"code": icd_code}
)[0]["label"]

q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
qwen_candidates = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 20, "qe": q_embed})

print(f"ICD {icd_code}: {icd_label}")
print(f"Qwen top-20 candidates retrieved.\n")

# Step 2: GNN reranks the same candidates
candidate_codes = [c['id'] for c in qwen_candidates]
gnn_result, _, _ = gnn_api.post('/compare', {
    'icd_code': icd_code,
    'candidate_codes': candidate_codes,
    'top_k': 10,
})

# Step 3: Side-by-side comparison
print(f"{'Rank':<6} {'Qwen':>6} {'GNN':>6} {'Delta':>6}  {'HPO Code':<12} {'Label'}")
print("-" * 80)
for c in gnn_result['by_gnn']:
    delta_str = f"+{c['rank_delta']}" if c['rank_delta'] > 0 else str(c['rank_delta'])
    print(f"  GNN {c['gnn_rank']:>2}  Q {c['qwen_rank']:>2}  {delta_str:>5}   {c['code']:<12} {c['label']}")

ICD M54.2: Cervicalgia
Qwen top-20 candidates retrieved.

Rank     Qwen    GNN  Delta  HPO Code     Label
--------------------------------------------------------------------------------
  GNN  1  Q 19    +18   HP:0030833   Neck pain
  GNN  2  Q  1     -1   HP:0008480   Cervical spondylosis
  GNN  3  Q  3      0   HP:0030009   Cervical insufficiency
  GNN  4  Q  9     +5   HP:0002947   Cervical kyphosis
  GNN  5  Q  6     +1   HP:0008445   Cervical spinal canal stenosis
  GNN  6  Q  2     -4   HP:6000107   Cervical motion tenderness
  GNN  7  Q 20    +13   HP:0000473   Torticollis
  GNN  8  Q  8      0   HP:0008462   Cervical instability
  GNN  9  Q  5     -4   HP:0003308   Cervical subluxation
  GNN 10  Q 12     +2   HP:0012318   Occipital neuralgia


### 5.2 Easy Case: R06.0 Dyspnoea -- GNN Preserves Correct Ranking

When Qwen already ranks the correct HPO at #1, the GNN preserves it (no harm).

In [16]:
# Easy case: R06.0 Dyspnoea
icd_code = "R06.0"

icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
    {"code": icd_code}
)[0]["label"]

q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
qwen_candidates = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 20, "qe": q_embed})

print(f"ICD {icd_code}: {icd_label}\n")

candidate_codes = [c['id'] for c in qwen_candidates]
gnn_result, _, _ = gnn_api.post('/compare', {
    'icd_code': icd_code,
    'candidate_codes': candidate_codes,
    'top_k': 10,
})

print(f"{'Rank':<6} {'Qwen':>6} {'GNN':>6} {'Delta':>6}  {'HPO Code':<12} {'Label'}")
print("-" * 80)
for c in gnn_result['by_gnn']:
    delta_str = f"+{c['rank_delta']}" if c['rank_delta'] > 0 else str(c['rank_delta'])
    print(f"  GNN {c['gnn_rank']:>2}  Q {c['qwen_rank']:>2}  {delta_str:>5}   {c['code']:<12} {c['label']}")

ICD R06.0: Dyspnoea

Rank     Qwen    GNN  Delta  HPO Code     Label
--------------------------------------------------------------------------------
  GNN  1  Q  1      0   HP:0002094   Dyspnea
  GNN  2  Q  3     +1   HP:0002789   Tachypnea
  GNN  3  Q  5     +2   HP:0002098   Respiratory distress
  GNN  4  Q 15    +11   HP:0002883   Hyperventilation
  GNN  5  Q  2     -3   HP:0002875   Exertional dyspnea
  GNN  6  Q  7     +1   HP:0012764   Orthopnea
  GNN  7  Q  9     +2   HP:0002093   Respiratory insufficiency
  GNN  8  Q 17     +9   HP:0002791   Hypoventilation
  GNN  9  Q  8     -1   HP:0005957   Breathing dysregulation
  GNN 10  Q 16     +6   HP:0005943   Respiratory arrest


### 5.3 GNN-Enhanced LLM Disambiguation

The GNN's value becomes clear in the LLM disambiguation step.
Without GNN reranking, "Neck pain" is not in the top-4 candidates presented to MedGemma.
With GNN reranking, it enters the candidate set and gets correctly selected.

In [17]:
# WITHOUT GNN: Qwen top-4 candidates for M54.2
icd_code = "M54.2"

icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
    {"code": icd_code}
)[0]["label"]

# ICD hierarchy context for the LLM
icd_context = graph_client.query("""
    MATCH (d:IcdDisease {id: $code})
    OPTIONAL MATCH (d)<-[:HAS_CHILD]-(parent:IcdDisease)
    OPTIONAL MATCH (d)<-[:GROUP_HAS_DISEASE]-(g:IcdGroup)
    OPTIONAL MATCH (g)<-[:CHAPTER_HAS_DISEASE]-(ch:IcdChapter)
    RETURN d.label AS name, parent.label AS parentName,
           g.label AS groupName, ch.label AS chapterName
    LIMIT 1
""", {"code": icd_code})[0]

# Qwen-only top-4 (no GNN)
q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
qwen_top4 = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 4, "qe": q_embed})

# Fetch HPO details for disambiguation
top4_codes = [c['id'] for c in qwen_top4]
hpo_details = graph_client.query("""
    UNWIND $codes AS code
    MATCH (h:HpoPhenotype {id: code})
    RETURN h.id AS id, h.label AS label,
           h.IAO_0000115 AS description, h.hasExactSynonym AS exactSynonym
""", {"codes": top4_codes})

candidate_list = []
for hpo in hpo_details:
    qwen_entry = next(c for c in qwen_top4 if c['id'] == hpo['id'])
    candidate_list.append({
        "score": qwen_entry["score"], "id": hpo["id"], "label": hpo["label"],
        "exactSynonym": hpo.get("exactSynonym", []) or [],
        "description": hpo.get("description", "") or "",
    })

payload = {
    "source_concept": f"{icd_code} {icd_label}",
    "source_context": json.dumps({
        "id": icd_code, "name": icd_context["name"],
        "parentName": icd_context.get("parentName", ""),
        "group": {"groupName": icd_context.get("groupName", "")},
        "chapter": {"chapterName": icd_context.get("chapterName", "")},
    }),
    "candidate_list": json.dumps(candidate_list),
}

print(f"Source: {icd_code} {icd_label}")
print(f"Qwen top-4 candidates (NO GNN -- 'Neck pain' is missing!):")
for c in candidate_list:
    print(f"  {c['score']:.4f}  {c['id']}  {c['label']}")

print("\nLLM disambiguation (picks from wrong candidates):")
tool_output = build_ontology_mapper_tool(chat_client).invoke(payload)
show_json(tool_output)

Source: M54.2 Cervicalgia
Qwen top-4 candidates (NO GNN -- 'Neck pain' is missing!):
  0.8813  HP:0008480  Cervical spondylosis
  0.8805  HP:6000107  Cervical motion tenderness
  0.8672  HP:0030009  Cervical insufficiency
  0.8569  HP:0032535  Cervical (neck)

LLM disambiguation (picks from wrong candidates):


/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OntologyMappingResponse(b...h the source concept.')), input_type=OntologyMappingResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "best_id": "HP:0008480",
  "best_label": "Cervical spondylosis",
  "confidence": 0.881343,
  "rationale": "Cervicalgia is a general term for neck pain. Cervical spondylosis is a degenerative condition affecting the cervical spine, which is a more specific and semantically equivalent concept.",
  "support": {
    "evidence": "Semantic equivalence and clinical relevance.",
    "reason": "Cervical spondylosis is a common cause of cervicalgia, and the description aligns with the source concept."
  }
}
```

In [18]:
# WITH GNN: reranked top-4 candidates for M54.2
icd_code = "M54.2"

icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
    {"code": icd_code}
)[0]["label"]

icd_context = graph_client.query("""
    MATCH (d:IcdDisease {id: $code})
    OPTIONAL MATCH (d)<-[:HAS_CHILD]-(parent:IcdDisease)
    OPTIONAL MATCH (d)<-[:GROUP_HAS_DISEASE]-(g:IcdGroup)
    OPTIONAL MATCH (g)<-[:CHAPTER_HAS_DISEASE]-(ch:IcdChapter)
    RETURN d.label AS name, parent.label AS parentName,
           g.label AS groupName, ch.label AS chapterName
    LIMIT 1
""", {"code": icd_code})[0]

# Qwen top-20 -> GNN rerank -> take top-4
q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
qwen_candidates = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 20, "qe": q_embed})

candidate_codes = [c['id'] for c in qwen_candidates]
gnn_reranked, _, _ = gnn_api.post('/rerank', {
    'icd_code': icd_code, 'candidate_codes': candidate_codes, 'top_k': 4,
})

top4_codes = [c['code'] for c in gnn_reranked['candidates']]
hpo_details = graph_client.query("""
    UNWIND $codes AS code
    MATCH (h:HpoPhenotype {id: code})
    RETURN h.id AS id, h.label AS label,
           h.IAO_0000115 AS description, h.hasExactSynonym AS exactSynonym
""", {"codes": top4_codes})

candidate_list = []
for hpo in hpo_details:
    gnn_entry = next(c for c in gnn_reranked['candidates'] if c['code'] == hpo['id'])
    candidate_list.append({
        "score": gnn_entry["gnn_score"], "id": hpo["id"], "label": hpo["label"],
        "exactSynonym": hpo.get("exactSynonym", []) or [],
        "description": hpo.get("description", "") or "",
    })

payload = {
    "source_concept": f"{icd_code} {icd_label}",
    "source_context": json.dumps({
        "id": icd_code, "name": icd_context["name"],
        "parentName": icd_context.get("parentName", ""),
        "group": {"groupName": icd_context.get("groupName", "")},
        "chapter": {"chapterName": icd_context.get("chapterName", "")},
    }),
    "candidate_list": json.dumps(candidate_list),
}

print(f"Source: {icd_code} {icd_label}")
print(f"GNN-reranked top-4 candidates ('Neck pain' is now present!):")
for c in candidate_list:
    print(f"  {c['score']:.4f}  {c['id']}  {c['label']}")

print("\nLLM disambiguation (now picks from correct candidates):")
tool_output = build_ontology_mapper_tool(chat_client).invoke(payload)
show_json(tool_output)

Source: M54.2 Cervicalgia
GNN-reranked top-4 candidates ('Neck pain' is now present!):
  0.6241  HP:0030833  Neck pain
  0.6202  HP:0008480  Cervical spondylosis
  0.5951  HP:0030009  Cervical insufficiency
  0.5887  HP:0002947  Cervical kyphosis

LLM disambiguation (now picks from correct candidates):


/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OntologyMappingResponse(b...aning of cervicalgia.")), input_type=OntologyMappingResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "best_id": "HP:0030833",
  "best_label": "Neck pain",
  "confidence": 0.7,
  "rationale": "Neck pain is a common symptom of cervicalgia and is semantically equivalent.",
  "support": {
    "evidence": "Neck pain is a common symptom of cervicalgia.",
    "reason": "The description of 'Neck pain' aligns with the clinical meaning of cervicalgia."
  }
}
```

### 5.4 Why GNN Reranking Works: Structural Context

The 2-layer GAT aggregates information from each node's 2-hop neighborhood:
- **Neck pain** (HP:0030833) -- 21 HpoDisease links, reaching 523 co-occurring phenotypes
- **Cervical spondylosis** (HP:0008480) -- only 4 HpoDisease links and 82 co-occurring phenotypes

Qwen captures lexical similarity ("cervical" -> "cervical spondylosis").
The GAT captures structural context (disease co-occurrence density), breaking the tie.

In [19]:
# Structural comparison: Neck pain vs Cervical spondylosis
icd_code = "M54.2"
correct_hpo = "HP:0030833"   # Neck pain (Qwen rank 19 -> GNN rank 1)
top_qwen_hpo = "HP:0008480"  # Cervical spondylosis (Qwen rank 1)

# HPO subClassOf hierarchy paths
for code, label, rank_info in [
    (correct_hpo, "Neck pain", "GNN rank 1, Qwen rank 19"),
    (top_qwen_hpo, "Cervical spondylosis", "Qwen rank 1"),
]:
    paths = graph_client.query("""
        MATCH path = (p:HpoPhenotype {id: $code})-[:subClassOf*]->(anc:HpoPhenotype)
        WHERE NOT (anc)-[:subClassOf]->()
        RETURN [n IN nodes(path) | n.label] AS hierarchy
    """, {"code": code})

    print(f"=== {code} ({label}) -- {rank_info} ===")
    for i, p in enumerate(paths):
        print(f"  {' -> '.join(p['hierarchy'])}")

# 2-hop neighborhood comparison (what the 2-layer GAT aggregates)
print("\n=== 2-Layer GAT Neighborhood Comparison ===")
for code, label in [(correct_hpo, "Neck pain"), (top_qwen_hpo, "Cervical spondylosis")]:
    stats = graph_client.query("""
        MATCH (p:HpoPhenotype {id: $code})
        OPTIONAL MATCH (d:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(p)
        OPTIONAL MATCH (d)-[:HAS_PHENOTYPIC_FEATURE]->(co_phe:HpoPhenotype)
        RETURN count(DISTINCT d) AS linked_diseases,
               count(DISTINCT co_phe) AS co_phenotypes_hop2
    """, {"code": code})[0]

    print(f"\n  {code} ({label}):")
    print(f"    Hop 1: {stats['linked_diseases']} HpoDisease links")
    print(f"    Hop 2: {stats['co_phenotypes_hop2']} co-occurring phenotypes (via shared diseases)")

=== HP:0030833 (Neck pain) -- GNN rank 1, Qwen rank 19 ===
  Neck pain -> Pain in head and neck region -> Pain -> Constitutional symptom -> Phenotypic abnormality -> All
=== HP:0008480 (Cervical spondylosis) -- Qwen rank 1 ===
  Cervical spondylosis -> Abnormal cervical spine morphology -> Abnormality of the cervical spine -> Abnormality of the vertebral column -> Abnormal axial skeleton morphology -> Abnormal skeletal morphology -> Abnormality of the skeletal system -> Abnormality of the musculoskeletal system -> Phenotypic abnormality -> All
  Cervical spondylosis -> Abnormal cervical spine morphology -> Abnormal vertebral morphology -> Abnormality of the vertebral column -> Abnormal axial skeleton morphology -> Abnormal skeletal morphology -> Abnormality of the skeletal system -> Abnormality of the musculoskeletal system -> Phenotypic abnormality -> All

=== 2-Layer GAT Neighborhood Comparison ===

  HP:0030833 (Neck pain):
    Hop 1: 21 HpoDisease links
    Hop 2: 523 co-occurring 

### 5.5 Fairness Note: Training vs. Inference Graph

At inference time, the GNN operates on the ontology hierarchy alone — no UMLS edges are present in the graph. The learned attention weights transfer structural patterns from training to a UMLS-free inference graph, confirming that the GNN captures genuine ontological structure rather than memorizing UMLS bridges.

## 6. GRetriever: LLM Disambiguation with Graph Tokens

The GRetriever injects graph structure as **soft tokens** directly into MedGemma:

```
ICD code + HPO candidates
    +-- PCST subgraph extraction (optimal connected subgraph)
    |       +-- GAT encoder -> per-node embeddings
    |               +-- MLP -> soft tokens [K, hidden_size]
    |                       |
    +-- [soft_tokens | text_prompt] -> MedGemma (LoRA) -> HPO code
```

**How GRetriever uses graph structure:**

1. **PCST subgraph selection** — the Prize-Collecting Steiner Tree algorithm selects an optimal
   connected subgraph around the ICD query node. Node prizes reflect relevance: ICD center = 5.0,
   HPO candidates = cosine_score x 4.0, context nodes = 0.1. Each edge costs 0.5. PCST maximizes
   total prizes minus total edge costs, keeping only structurally justified connections.
   Candidates not worth connecting are force-included but remain disconnected.

2. **GAT encoding** — a 2-layer Graph Attention Network runs message passing on the PCST subgraph,
   producing per-node embeddings that capture structural context (connected nodes get richer
   representations than disconnected ones).

3. **Soft token injection** — an MLP projects each key node's embedding into one soft token
   (dim=2048), prepended to the text prompt.

4. **LoRA fine-tuning** — MedGemma's attention layers (`q_proj`, `v_proj`) are jointly trained
   with the GNN+MLP via low-rank adaptation (rank=8, ~0.1% of LLM parameters). This is essential:
   without LoRA, the frozen LLM cannot interpret the soft tokens (59% accuracy). LoRA teaches
   the attention layers to read graph structure from the prepended tokens, while the end-to-end
   training (cross-entropy loss → LoRA → MLP → GAT) ensures the GNN produces tokens the LLM
   can actually use. LoRA alone already learns the disambiguation task well (74.4% text-only);
   the graph tokens provide complementary structural signal on top.

### Aggregate Results

| Method | Accuracy |
|--------|----------|
| Cosine top-1 (baseline) | 68.9% |
| GRetriever+PCST+LoRA | 74.1% |

**Key finding:** LoRA fine-tuning is the dominant improvement (+5.5% over cosine baseline).
Graph tokens add marginal aggregate value, but are complementary to the GNN reranker
for different downstream tasks (see Section 7.4).


### 6.1 R06.3 Periodic Breathing — PCST Subgraph in Action

R06.3 (Periodic breathing) is a pattern of cyclically varying ventilation depth.
The cosine top-10 includes both the correct answer (Cheyne-Stokes respiration) and
close distractors (Intermittent hyperventilation, Periodic breathing, etc.).

This example shows the full PCST subgraph: which candidates get connected,
which remain disconnected, and how the graph context changes the prediction.


In [20]:
# R06.3 Periodic breathing: text-only vs GRetriever + PCST subgraph
candidates = [
    'HP:0004879', 'HP:0012196', 'HP:0012195', 'HP:0005957', 'HP:0002793',
    'HP:0005941', 'HP:0030207', 'HP:0004881', 'HP:0040213', 'HP:0002877',
]
result, _, _ = gret_api.post('/compare', {
    'icd_code': 'R06.3',
    'candidate_codes': candidates,
})

text_pred = result['text_only']
gret_pred = result['gretriever']
sub = result['subgraph']
viz = sub['viz']

print(f"ICD R06.3: {result['icd_label']}")
print(f"\n{'='*70}")
print(f"  Text-only (LoRA, no graph):  {text_pred['code']}  {text_pred['name']}")
print(f"  GRetriever (LoRA + graph):   {gret_pred['code']}  {gret_pred['name']}")
print(f"  Correct answer:              HP:0012196  Cheyne-Stokes respiration")
print(f"  Subgraph: {sub['subgraph_nodes']} nodes, {sub['subgraph_edges']} edges")
print(f"{'='*70}")

# Display PCST subgraph structure
node_map = {n['id']: n for n in viz['nodes']}
connected_ids = set()
for e in viz['edges']:
    connected_ids.add(e['source'])
    connected_ids.add(e['target'])

print(f"\nPCST subgraph — connected edges (cost=0.5 each):")
seen = set()
for e in viz['edges']:
    key = (min(e['source'], e['target']), max(e['source'], e['target']))
    if key in seen:
        continue
    seen.add(key)
    s = node_map[e['source']]
    t = node_map[e['target']]
    print(f"    ({s['name']} p={s['prize']:.1f})-[cost=0.5]->({t['name']} p={t['prize']:.1f})")

disconnected = [n for n in viz['nodes'] if n['id'] not in connected_ids]
if disconnected:
    print(f"\n  Force-included candidates (not connected by PCST):")
    for n in disconnected:
        print(f"    ({n['name']} p={n['prize']:.1f})")


ICD R06.3: Periodic breathing

  Text-only (LoRA, no graph):  HP:0004879  Intermittent hyperventilation
  GRetriever (LoRA + graph):   HP:0012196  Cheyne-Stokes respiration
  Correct answer:              HP:0012196  Cheyne-Stokes respiration
  Subgraph: 12 nodes, 6 edges

PCST subgraph — connected edges (cost=0.5 each):
    (Abnormalities of breathing p=0.1)-[cost=0.5]->(Periodic breathing p=5.0)
    (Abnormal pattern of respiration p=2.7)-[cost=0.5]->(Irregular respiration p=2.8)
    (Abnormal pattern of respiration p=2.7)-[cost=0.5]->(Cheyne-Stokes respiration p=2.9)
    (Abnormal pattern of respiration p=2.7)-[cost=0.5]->(Paradoxical respiration p=2.7)
    (Abnormal pattern of respiration p=2.7)-[cost=0.5]->(Hypopnea p=2.5)
    (Intermittent hyperventilation p=2.9)-[cost=0.5]->(Intermittent hyperpnea at rest p=2.7)

  Force-included candidates (not connected by PCST):
    (Breathing dysregulation p=2.8)
    (Nocturnal hypoventilation p=2.5)
    (Episodic hypoventilation p=2.6)


## 7. Patient Annotation

### 7.1 Named Entity Recognition (NER)

Extract clinical entities from concatenated encounter text, classifying them by
ICD chapter, assertion status, and temporality.

In [21]:
# Patient NER: extract clinical entities from encounter text
payload = {
    "patient_id": "patient_002",
    "encounter_id": "enc_145",
    "icd_chapters": [
        "Certain infectious and parasitic diseases", "Neoplasms",
        "Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism",
        "Endocrine, nutritional and metabolic diseases", "Mental and behavioural disorders",
        "Diseases of the nervous system", "Diseases of the eye and adnexa",
        "Diseases of the ear and mastoid process", "Diseases of the circulatory system",
        "Diseases of the respiratory system", "Diseases of the digestive system",
        "Diseases of the skin and subcutaneous tissue",
        "Diseases of the musculoskeletal system and connective tissue",
        "Diseases of the genitourinary system", "Pregnancy, childbirth and the puerperium",
        "Certain conditions originating in the perinatal period",
        "Congenital malformations, deformations and chromosomal abnormalities",
        "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified",
        "Injury, poisoning and certain other consequences of external causes",
        "External causes of morbidity and mortality",
        "Factors influencing health status and contact with health services",
        "Codes for special purposes",
    ],
    "concat_text": (
        "Abdominal pain | Shortness of breath and chest tightness | "
        "Episodic dizziness and palpitations | "
        "Hypertension, type 2 diabetes mellitus, and iron-deficiency anemia | No Chest pain"
    ),
    "narrative_text": "No narrative text provided.",
}

tool_output = build_patient_ner_tool(chat_client).invoke(payload)
show_json(tool_output)

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PatientNERResponse(patien...should be extracted.')]), input_type=PatientNERResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "patient_id": "patient_002",
  "encounter_id": "enc_145",
  "entities": [
    {
      "source": "concat",
      "start": 13,
      "end": 26,
      "text": "Abdominal pain",
      "label": "Diseases of the digestive system",
      "assertion": "present",
      "temporality": "unspecified",
      "rationale": "Present in the concatenated text."
    },
    {
      "source": "concat",
      "start": 30,
      "end": 46,
      "text": "Shortness of breath and chest tightness",
      "label": "Diseases of the respiratory system",
      "assertion": "present",
      "temporality": "unspecified",
      "rationale": "Present in the concatenated text."
    },
    {
      "source": "concat",
      "start": 50,
      "end": 72,
      "text": "Episodic dizziness and palpitations",
      "label": "Diseases of the nervous system",
      "assertion": "present",
      "temporality": "unspecified",
      "rationale": "Present in the concatenated text."
    },
    {
      "source": "concat",
      "start": 77,
      "end": 106,
      "text": "Hypertension, type 2 diabetes mellitus, and iron-deficiency anemia",
      "label": "Endocrine, nutritional and metabolic diseases",
      "assertion": "present",
      "temporality": "unspecified",
      "rationale": "Present in the concatenated text."
    },
    {
      "source": "concat",
      "start": 111,
      "end": 124,
      "text": "No Chest pain",
      "label": "Diseases of the circulatory system",
      "assertion": "negated",
      "temporality": "unspecified",
      "rationale": "The patient denies chest pain."
    },
    {
      "source": "narrative",
      "start": 0,
      "end": 13,
      "text": "No narrative text provided.",
      "label": "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified",
      "assertion": "unspecified",
      "temporality": "unspecified",
      "rationale": "No narrative text provided. The absence of a narrative does not preclude the presence of symptoms or signs. The concatenated text contains abdominal pain. This is a symptom and therefore should be extracted."
    }
  ]
}
```

Example of query to show connections between virtual and materialized nodes:

### 7.2 Named Entity Disambiguation (NED)

Given an extracted mention and candidate ICD codes, select the best match
considering surrounding clinical context.

In [22]:
# Patient NED: disambiguate "terminal ileitis" to the correct ICD code
ned_payload = {
    "mention": {
        "source": "concat", "start": 245, "end": 260,
        "text": "terminal ileitis",
        "label": "Diseases of the digestive system",
        "assertion": "present", "temporality": "chronic",
        "rationale": "Imaging and colonoscopy describe inflammation of the terminal ileum.",
    },
    "candidates": [
        {"score": 0.912, "id": "K52.9", "label": "Noninfective gastroenteritis and colitis, unspecified"},
        {"score": 0.887, "id": "A09.0", "label": "Infectious gastroenteritis and colitis, unspecified"},
        {"score": 0.871, "id": "K52.0", "label": "Gastroenteritis and colitis due to radiation"},
        {"score": 0.842, "id": "K50.00", "label": "Crohn's disease of small intestine without complications"},
        {"score": 0.824, "id": "K50.80", "label": "Crohn's disease of both small and large intestine without complications"},
        {"score": 0.801, "id": "K51.90", "label": "Ulcerative colitis, unspecified, without complications"},
    ],
    "other_mentions": [
        {"text": "long-standing Crohn disease diagnosed at age 19", "label": "Diseases of the digestive system"},
        {"text": "skip lesions in terminal ileum and ascending colon on colonoscopy", "label": "Diseases of the digestive system"},
        {"text": "chronic watery diarrhea and 7 kg unintentional weight loss in 6 months", "label": "Symptoms, signs and abnormal clinical and laboratory findings"},
        {"text": "non-caseating granulomas on ileal biopsy", "label": "Diseases of the digestive system"},
    ],
}

ned_tool_output = build_patient_ned_tool(chat_client).invoke(ned_payload)
show_json(ned_tool_output)

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PatientNEDResponse(source... for terminal ileitis.'), input_type=PatientNEDResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "source": "concat",
  "start": 245,
  "end": 260,
  "text": "terminal ileitis",
  "label": "Diseases of the digestive system",
  "assertion": "present",
  "temporality": "chronic",
  "rationale": "The mention describes inflammation of the terminal ileum, which aligns with the ICD code for terminal ileitis.",
  "icd_id": "K50.00",
  "icd_label": "Crohn's disease of small intestine without complications",
  "confidence": 0.912,
  "linking_rationale": "The mention describes inflammation of the terminal ileum, which aligns with the ICD code for terminal ileitis."
}
```

Query to show connection between virtual and materialized nodes
```
CALL apoc.dv.query('encounter', {patientId: 'P003'}) YIELD node AS v
WITH v, split(
  replace(replace(replace(replace(v.ICD10_Codes, '[', ''), ']', ''), "'", ''), ' ', ''),
  ','
) AS icd_codes
LIMIT 1
UNWIND icd_codes AS icd_code
WITH v, icd_code WHERE icd_code <> ''
MATCH (d:IcdDisease {id: icd_code})-[r:ICD_MAPS_TO_HPO_BY_EMBEDDING]->(p:HpoPhenotype)
RETURN v, apoc.create.vRelationship(v, 'HAS_ICD_CODE', {}, d) AS vr, d, r, p
LIMIT 10
```

## 8. GraphRAG Agent

The LangGraph agent routes questions through different pipelines:
- **Ontology queries** -> text2cypher (LLM-generated Cypher)
- **Patient info** -> APOC data virtualization
- **Disease coverage** -> ICD->HPO->ancestors->coverage pipeline

In [23]:
from llm.agent import run_agent

In [24]:
# Ontology query: find phenotypes for a disease
out = run_agent("Find phenotypes associated with the avian influenza. ")
print(out["final_answer"])

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GuardrailsDecision(decisi...nt topic for this app.'), input_type=GuardrailsDecision])
  return self.__pydantic_serializer__.to_python(
/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ValidateCypherOutput(errors=[]), input_type=ValidateCypherOutput])
  return self.__pydantic_serializer__.to_python(


Avian influenza is associated with a variety of phenotypes, including abdominal pain, acute kidney injury, chest pain, congestive heart failure, conjunctivitis, cough, decreased total leukocyte count, decreased total lymphocyte count, diarrhea, disseminated intravascular coagulation, dyspnea, elevated circulating C-reactive protein concentration, elevated circulating creatine kinase concentration, elevated circulating hepatic transaminase concentration, fatigue, fever, ground-glass opacification, headache, hepatitis, hypoalbuminemia, hypoxemia, increased circulating lactate dehydrogenase concentration, infectious encephalitis, meningitis, miscarriage, myalgia, myelitis, nonproductive cough, productive cough, rhabdomyolysis, sepsis, tachypnea and thrombocytopenia.



In [25]:
# LLM chat completion with MedGemma
client = OpenAI(api_key="EMPTY", base_url=url_llm)
resp = client.chat.completions.create(
    model="google/medgemma-4b-it",
    messages=[{"role": "user", "content": (
        "Find phenotypes associated with the avian influenza. Is the following answer correct? " + out["final_answer"]
    )}],
    temperature=0,
)
print(f"LLM: {resp.choices[0].message.content}")

LLM: The answer you provided is **correct**. It lists a comprehensive and accurate range of phenotypes associated with avian influenza (bird flu).

Here's why it's a good answer:

*   **Variety of Systems Affected:** The list covers a wide array of organ systems, including the respiratory, cardiovascular, renal, hepatic, hematologic, neurologic, and reproductive systems.
*   **Clinical Signs:** It includes both clinical signs (e.g., cough, dyspnea, fever, headache) and laboratory findings (e.g., decreased leukocyte count, elevated CRP, elevated creatine kinase).
*   **Severity Spectrum:** The phenotypes range from mild (e.g., conjunctivitis, myalgia) to severe (e.g., acute kidney injury, disseminated intravascular coagulation, sepsis, death).
*   **Common Findings:** Many of the listed phenotypes are commonly observed in avian influenza infections.

Therefore, the answer is accurate and provides a good overview of the phenotypic manifestations of avian influenza.



In [26]:
# Patient info query: clinical summary via data virtualization
out = run_agent("Provide me details on the follow up plan of patient P003 in the latest encounter.")
print(out["final_answer"])

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GuardrailsDecision(decisi...the scope of this app.'), input_type=GuardrailsDecision])
  return self.__pydantic_serializer__.to_python(


Answer:
The follow-up plan for patient P003 in the latest encounter on 2026-03-03 is to monitor for new skin changes and reassess if the lesions spread. The patient is instructed to use regular emollients.



In [27]:
# Coverage query: disease ranking by phenotype coverage
out = run_agent("Show possible diseases by HPO coverage for patientId:'P003'. Report the covered, total, and percentage coverage.")
print(out["final_answer"])

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GuardrailsDecision(decisi... the scope of the app.'), input_type=GuardrailsDecision])
  return self.__pydantic_serializer__.to_python(


DEBUG: Retrieved ICD codes: ['A68', 'A68.9', 'B48.8', 'F40.0', 'F43.8', 'F51.4', 'F59', 'F90.0', 'G02.1', 'G03.0', 'G44', 'G44.2', 'G57.1', 'G58.7', 'H30.0', 'H30.2', 'H35.0', 'H43.0', 'H49.1', 'H49.2', 'H49.3', 'H50.2', 'H53.2', 'H53.9', 'H57.1', 'I44.0', 'I47', 'I48.4', 'I49.4', 'K03.0', 'K07.2', 'K07.6', 'L90.6', 'M13.1', 'M23.4', 'M24.5', 'M25.4', 'M25.5', 'M25.6', 'M54.2', 'M54.8', 'R00.2', 'R11', 'R22.4', 'R23.8', 'R25.0', 'R26.0', 'R26.2', 'R41.1', 'R41.8', 'R42', 'R50', 'R50.8', 'R51', 'R52', 'R52.9', 'R53', 'R55', 'R70.0', 'R83.4', 'S03.4', 'S13.4', 'S84', 'S94.7', 'T03.0', 'T28.1', 'W10', 'Z50']
Based on the provided data, the following diseases are covered for patientId 'P003':

*   Developmental delay, impaired speech, and behavioral abnormalities (OMIM:619475) - 18/188 (9.6%)
*   Behçet disease (ORPHA:117) - 17/85 (20.0%)
*   Brucellosis (ORPHA:1304) - 16/77 (20.8%)
*   Lyme disease (ORPHA:900) - 16/24 (66.7%)
*   Giant cell arteritis (ORPHA:289390) - 15/69 (21.7%)
*   Afr

## 9. Three-Way Pipeline Comparison: Patient P003

Same patient, same downstream pipeline, three different ICD-to-HPO mapping methods.
Differences in mapping quality propagate through ancestor rollup, disease coverage,
and into the agent's final clinical answer.

```
Patient ICD codes (68)
    +-- [A] Qwen embedding cosine top-1
    +-- [B] Qwen + GNN reranker top-1          -> HPO -> ancestors -> coverage -> agent answer
    +-- [C] Qwen + GRetriever (LoRA + graph tokens)
```

### 9.1 Step 1: ICD-to-HPO Mapping (3 Methods)

Map all 68 ICD codes through each method. Track where they disagree.

In [28]:
# Three-Way ICD->HPO Mapping -- Real Patient P003
from llm.query_factory import (
    get_patient_icd_codes, rollup_hpo_to_ancestors, compute_coverage, _run_query
)

patient_id = "P003"
patient_icd = get_patient_icd_codes(patient_id)
print(f"Patient {patient_id}: {len(patient_icd)} ICD codes")
print(f"  Sample: {patient_icd[:10]}...\n")

print("=" * 90)
print("STEP 1: ICD -> HPO Mapping (3 methods)")
print("=" * 90)

mappings = {"qwen": {}, "gnn": {}, "gretriever": {}}
diffs = {"gnn_vs_qwen": 0, "gret_vs_qwen": 0, "gret_vs_gnn": 0}

for i, icd_code in enumerate(patient_icd):
    if i % 10 == 0:
        print(f"  Mapping... {i}/{len(patient_icd)}")

    icd_label_rows = graph_client.query(
        "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
        {"code": icd_code}
    )
    if not icd_label_rows:
        continue
    icd_label = icd_label_rows[0]["label"]

    # [A] Qwen cosine top-1
    q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
    qwen_top = graph_client.query("""
        CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
        YIELD node, score
        RETURN node.id AS id, node.label AS label, score
        ORDER BY score DESC LIMIT $k
    """, {"k": 20, "qe": q_embed})
    if not qwen_top:
        continue
    candidate_codes = [c['id'] for c in qwen_top]
    mappings["qwen"][icd_code] = {"id": qwen_top[0]["id"], "label": qwen_top[0]["label"]}

    # [B] GNN reranker top-1
    try:
        gnn_result, _, _ = gnn_api.post('/rerank', {
            'icd_code': icd_code, 'candidate_codes': candidate_codes, 'top_k': 1,
        })
        gnn_top1 = gnn_result['candidates'][0]
        mappings["gnn"][icd_code] = {"id": gnn_top1["code"], "label": gnn_top1["label"]}
    except Exception:
        mappings["gnn"][icd_code] = mappings["qwen"][icd_code]

    # [C] GRetriever (LoRA + graph tokens)
    try:
        gret_result, _, _ = gret_api.post('/disambiguate', {
            'icd_code': icd_code, 'candidate_codes': candidate_codes,
        })
        gret_pred = gret_result['prediction']
        mappings["gretriever"][icd_code] = {"id": gret_pred["code"], "label": gret_pred["name"]}
    except Exception:
        mappings["gretriever"][icd_code] = mappings["qwen"][icd_code]

    # Track differences
    q = mappings["qwen"][icd_code]["id"]
    g = mappings["gnn"][icd_code]["id"]
    r = mappings["gretriever"][icd_code]["id"]
    if g != q: diffs["gnn_vs_qwen"] += 1
    if r != q: diffs["gret_vs_qwen"] += 1
    if r != g: diffs["gret_vs_gnn"] += 1

n_mapped = len(mappings["qwen"])
print(f"\n  Mapped: {n_mapped} / {len(patient_icd)} ICD codes")
print(f"  GNN differs from Qwen:       {diffs['gnn_vs_qwen']} / {n_mapped}")
print(f"  GRetriever differs from Qwen: {diffs['gret_vs_qwen']} / {n_mapped}")
print(f"  GRetriever differs from GNN:  {diffs['gret_vs_gnn']} / {n_mapped}")

# Show cases where all 3 methods disagree
print(f"\n  Cases where all 3 methods disagree:")
print(f"  {'ICD':<8} {'Qwen':<30} {'GNN':<30} {'GRetriever':<30}")
print(f"  {'-'*98}")
for icd_code in mappings["qwen"]:
    q = mappings["qwen"][icd_code]
    g = mappings["gnn"].get(icd_code, q)
    r = mappings["gretriever"].get(icd_code, q)
    if q["id"] != g["id"] and q["id"] != r["id"] and g["id"] != r["id"]:
        print(f"  {icd_code:<8} {q['label'][:28]:<30} {g['label'][:28]:<30} {r['label'][:28]:<30}")

Patient P003: 68 ICD codes
  Sample: ['A68', 'A68.9', 'B48.8', 'F40.0', 'F43.8', 'F51.4', 'F59', 'F90.0', 'G02.1', 'G03.0']...

STEP 1: ICD -> HPO Mapping (3 methods)
  Mapping... 0/68
  Mapping... 10/68
  Mapping... 20/68
  Mapping... 30/68
  Mapping... 40/68
  Mapping... 50/68
  Mapping... 60/68

  Mapped: 68 / 68 ICD codes
  GNN differs from Qwen:       40 / 68
  GRetriever differs from Qwen: 30 / 68
  GRetriever differs from GNN:  37 / 68

  Cases where all 3 methods disagree:
  ICD      Qwen                           GNN                            GRetriever                    
  --------------------------------------------------------------------------------------------------
  B48.8    Phaeohyphomycosis              Coccidioidomycosis             Chronic mucocutaneous candid  
  F43.8    Intense psychological distre   Panic attack                   Dissociation                  
  G57.1    Acroparesthesia                Paresthesia                    Greater auricular nerve thic

### 9.2 Step 2: Agent Integration

Run the full LangGraph agent with each mapping backend. The agent executes:
`guardrails -> extract -> get_icd -> icd_to_hpo -> rollup -> coverage -> finalize`

We monkey-patch `map_icd_to_hpo` in both `query_factory` and `agent` modules
to swap the mapping method at runtime.

In [29]:
# Agent with 3 different mapping backends
from llm.agent import build_graph, AgentState
import llm.query_factory as qf
import llm.agent as agent_mod

_original_qf = qf.map_icd_to_hpo
_original_agent = agent_mod.map_icd_to_hpo


def make_mapper(method_name, gnn_api_client=None, gret_api_client=None, emb_api=None, graph=None):
    """Return a map_icd_to_hpo function that uses the specified method."""
    def mapper(icd_codes):
        hpo_ids = []
        for icd_code in icd_codes:
            icd_label_rows = graph.query(
                "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label", {"code": icd_code}
            )
            if not icd_label_rows:
                continue
            icd_label = icd_label_rows[0]["label"]

            q_embed = emb_api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
            candidates = graph.query("""
                CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
                YIELD node, score
                RETURN node.id AS id, node.label AS label, score
                ORDER BY score DESC LIMIT $k
            """, {"k": 20, "qe": q_embed})
            if not candidates:
                continue
            candidate_codes = [c['id'] for c in candidates]

            if method_name == "qwen":
                hpo_ids.append(candidates[0]["id"])
            elif method_name == "gnn":
                try:
                    result, _, _ = gnn_api_client.post('/rerank', {
                        'icd_code': icd_code, 'candidate_codes': candidate_codes, 'top_k': 1,
                    })
                    hpo_ids.append(result['candidates'][0]['code'])
                except Exception:
                    hpo_ids.append(candidates[0]["id"])
            elif method_name == "gretriever":
                try:
                    result, _, _ = gret_api_client.post('/disambiguate', {
                        'icd_code': icd_code, 'candidate_codes': candidate_codes,
                    })
                    hpo_ids.append(result['prediction']['code'])
                except Exception:
                    hpo_ids.append(candidates[0]["id"])
        return list(set(hpo_ids))
    return mapper


question = f"Show possible diseases by HPO coverage for patientId:'{patient_id}'. Report the covered, total, and percentage coverage."

agent_results = {}

print("=" * 90)
print(f"STEP 2: Agent Final Answers -- Patient {patient_id}")
print("=" * 90)

for method_name, label in [("qwen", "[A] Qwen-only"), ("gnn", "[B] GNN reranker"), ("gretriever", "[C] GRetriever")]:
    mapper = make_mapper(method_name, gnn_api_client=gnn_api, gret_api_client=gret_api, emb_api=api, graph=graph_client)
    qf.map_icd_to_hpo = mapper
    agent_mod.map_icd_to_hpo = mapper

    try:
        compiled = build_graph()
        result = compiled.invoke({
            "question": question, "patient_id": patient_id, "steps": [], "mode": "stepwise",
        })
        agent_results[method_name] = result
        n_hpo = len(result.get('hpo_ids') or [])
        print(f"\n{label}")
        print(f"  Steps: {result.get('steps')}")
        print(f"  HPO phenotypes mapped: {n_hpo}")
        print(f"\n  Final answer (first 600 chars):")
        print(f"  {result.get('final_answer', 'No answer')[:600]}")
        print(f"  {'---' * 27}")
    except Exception as e:
        print(f"\n{label}: ERROR -- {e}")

# Restore original functions
qf.map_icd_to_hpo = _original_qf
agent_mod.map_icd_to_hpo = _original_agent
print(f"\nOriginal map_icd_to_hpo restored.")

STEP 2: Agent Final Answers -- Patient P003
DEBUG: Retrieved ICD codes: ['A68', 'A68.9', 'B48.8', 'F40.0', 'F43.8', 'F51.4', 'F59', 'F90.0', 'G02.1', 'G03.0', 'G44', 'G44.2', 'G57.1', 'G58.7', 'H30.0', 'H30.2', 'H35.0', 'H43.0', 'H49.1', 'H49.2', 'H49.3', 'H50.2', 'H53.2', 'H53.9', 'H57.1', 'I44.0', 'I47', 'I48.4', 'I49.4', 'K03.0', 'K07.2', 'K07.6', 'L90.6', 'M13.1', 'M23.4', 'M24.5', 'M25.4', 'M25.5', 'M25.6', 'M54.2', 'M54.8', 'R00.2', 'R11', 'R22.4', 'R23.8', 'R25.0', 'R26.0', 'R26.2', 'R41.1', 'R41.8', 'R42', 'R50', 'R50.8', 'R51', 'R52', 'R52.9', 'R53', 'R55', 'R70.0', 'R83.4', 'S03.4', 'S13.4', 'S84', 'S94.7', 'T03.0', 'T28.1', 'W10', 'Z50']

[A] Qwen-only
  Steps: ['guardrails', 'extract', 'get_icd', 'icd_to_hpo', 'rollup', 'coverage', 'finalize']
  HPO phenotypes mapped: 66

  Final answer (first 600 chars):
  Based on the provided data, the following diseases are covered for patientId 'P003':

*   Behçet disease (ORPHA:117) - 17/85 (20.0%)
*   Developmental delay, impaired sp

In [30]:
# Disease disagreements across methods (top-N)
TOP_N = 15

disease_sets = {}  # method -> set of diseaseIds
disease_names = {}  # diseaseId -> diseaseName

for method_name in ["qwen", "gnn", "gretriever"]:
    res = agent_results.get(method_name)
    if not res or not res.get("results"):
        continue
    disease_sets[method_name] = set()
    for row in res["results"][:TOP_N]:
        did = row["diseaseId"]
        disease_sets[method_name].add(did)
        disease_names[did] = row["diseaseName"]

methods = list(disease_sets.keys())
all_diseases = set.union(*disease_sets.values())
shared = set.intersection(*disease_sets.values())
disagreements = all_diseases - shared

print(f"Disease disagreements (top-{TOP_N})")
print("=" * 90)
print(f"{'Disease':<55} {'Qwen':>10} {'GNN':>10} {'GRet':>10}")
print("-" * 90)

for did in sorted(disagreements, key=lambda d: disease_names[d]):
    present = ["yes" if did in disease_sets[m] else "-" for m in methods]
    name = disease_names[did][:53]
    print(f"  {name:<53} {present[0]:>10} {present[1]:>10} {present[2]:>10}")

print(f"\n  Shared across all methods: {len(shared)}/{len(all_diseases)}")
print(f"  Disagreements:             {len(disagreements)}/{len(all_diseases)}")


Disease disagreements (top-15)
Disease                                                       Qwen        GNN       GRet
------------------------------------------------------------------------------------------
  17q11 microdeletion syndrome                                   -          -        yes
  African trypanosomiasis                                      yes        yes          -
  Amoebiasis due to free-living amoebae                        yes          -        yes
  Cysticercosis                                                yes          -        yes
  Marchiafava-Bignami disease                                  yes          -        yes
  Meningioma                                                     -          -        yes
  Multiple mitochondrial dysfunctions syndrome 9B              yes        yes          -
  Osteogenesis imperfecta                                        -        yes          -
  Plague                                                         -        yes

In [31]:
# Step 4: LLM-as-Judge — MedGemma evaluates disease coverage disagreements
from llm.pipeline_patient import get_patient_views

print("=" * 90)
print(f"STEP 4: LLM-as-Judge — MedGemma evaluates disease disagreements for Patient {patient_id}")
print("=" * 90)

# Fetch patient clinical notes via APOC Data Virtualization
views = get_patient_views(patient_id)
if views:
    encounter_sections = []
    for v in views:
        enc_id = v.get("encounter", {}).get("id", "unknown")
        enc_date = v.get("encounter", {}).get("period_start", "")
        section = "\n".join(filter(None, [
            f"--- Encounter {enc_id} ({enc_date}) ---",
            f"Condition: {v.get('condition', '')}" if v.get('condition') else None,
            f"Chief complaint: {v.get('chief_complaint', '')}" if v.get('chief_complaint') else None,
            f"Course/trend: {v.get('course_trend', '')}" if v.get('course_trend') else None,
            f"Comorbidities: {v.get('comorbidities', '')}" if v.get('comorbidities') else None,
            f"Narrative: {v.get('narrative', '')}" if v.get('narrative') else None,
            f"Observations: {v.get('observation_text', '')}" if v.get('observation_text') else None,
            f"Diagnostic report: {v.get('diagnostic_report', '')}" if v.get('diagnostic_report') else None,
            f"Plan/follow-up: {v.get('plan_followup', '')}" if v.get('plan_followup') else None,
        ]))
        encounter_sections.append(section)
    clinical_notes = "\n\n".join(encounter_sections)
else:
    clinical_notes = "No clinical data available."

print(f"\n  {len(disagreements)} diseases where methods disagree\n")

for did in sorted(disagreements, key=lambda d: disease_names[d]):
    name = disease_names[did]
    present_in = [m for m in methods if did in disease_sets[m]]
    absent_in = [m for m in methods if did not in disease_sets[m]]

    question = (
        f"Clinical notes:\n{clinical_notes}\n\n"
        f"Disease: {name} ({did})\n\n"
        f"Based on the clinical notes above, is this disease a plausible "
        f"differential diagnosis for this patient? "
        f"Answer YES or NO, then explain why."
    )

    resp = client.chat.completions.create(
        model="google/medgemma-4b-it",
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    verdict = resp.choices[0].message.content.strip()

    method_map = {"qwen": "Qwen", "gnn": "GNN", "gretriever": "GRet"}
    present_str = ", ".join(method_map.get(m, m) for m in present_in)
    absent_str = ", ".join(method_map.get(m, m) for m in absent_in)

    print(f"  {name}")
    print(f"    Detected by: {present_str}  |  Missed by: {absent_str}")
    print(f"    MedGemma: {verdict[:300]}")
    print()


STEP 4: LLM-as-Judge — MedGemma evaluates disease disagreements for Patient P003

  11 diseases where methods disagree

  17q11 microdeletion syndrome
    Detected by: GRet  |  Missed by: Qwen, GNN
    MedGemma: YES

Explanation:

The clinical notes describe a patient with a complex medical history including multiple mononeuropathy, aseptic meningitis, bruxism, posterior uveitis, paroxysmal tachycardia, transient memory impairment, tension-type headache, recurrent monoarthritis, recurrent low-grade fever, f

  African trypanosomiasis
    Detected by: Qwen, GNN  |  Missed by: GRet
    MedGemma: NO.

Explanation:

While the patient has a history of multiple mononeuropathy, posterior uveitis, paroxysmal tachycardia, recurrent fever, and first-degree AV block, none of these conditions are directly associated with African trypanosomiasis (sleeping sickness). African trypanosomiasis is a parasi

  Amoebiasis due to free-living amoebae
    Detected by: Qwen, GRet  |  Missed by: GNN
    MedGem

### 9.3 Complementary Strengths: GNN vs GRetriever

The GNN reranker and GRetriever optimize for **different objectives**:
- **GNN** picks broad, well-connected HPO terms -> maximizes disease coverage (screening)
- **GRetriever** picks precise, clinically specific terms -> maximizes phenotype accuracy (annotation)

The R42 (Dizziness) case illustrates this: GNN picks "Nausea" (wrong symptom but broad),
GRetriever picks "Vertigo" (clinically correct).

In [32]:
# R42 (Dizziness): GNN vs GRetriever -- Different Strengths
icd_code = "R42"
icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label", {"code": icd_code}
)[0]["label"]

print(f"=== ICD {icd_code}: {icd_label} ===\n")

# Get the mappings from Step 1
q = mappings["qwen"][icd_code]
g = mappings["gnn"][icd_code]
r = mappings["gretriever"][icd_code]

print(f"  Qwen:       {q['id']}  {q['label']}")
print(f"  GNN:        {g['id']}  {g['label']}")
print(f"  GRetriever: {r['id']}  {r['label']}")

print(f"\n  Clinical assessment:")
print(f"    Qwen 'Paroxysmal vertigo' -- too specific (implies episodic BPPV)")
print(f"    GNN  'Nausea' -- wrong symptom (nausea != dizziness)")
print(f"    GRetriever 'Vertigo' -- correct (dizziness = vertigo)")

# Disease coverage impact
print(f"\n  Disease coverage impact:")
for method, m, label in [("Qwen", q, "[A]"), ("GNN", g, "[B]"), ("GRetriever", r, "[C]")]:
    n_diseases = graph_client.query("""
        MATCH (d:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(p:HpoPhenotype {id: $hpo})
        RETURN count(d) AS n
    """, {"hpo": m["id"]})[0]["n"]

    n_ancestor_diseases = graph_client.query("""
        MATCH (p:HpoPhenotype {id: $hpo})-[:subClassOf*0..]->(anc:HpoPhenotype)
        WITH collect(DISTINCT anc) AS ancestors
        UNWIND ancestors AS a
        MATCH (d:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(a)
        RETURN count(DISTINCT d) AS n
    """, {"hpo": m["id"]})[0]["n"]

    print(f"    {label} {method:<12} {m['id']}  -> {n_diseases:>4} direct diseases, {n_ancestor_diseases:>5} via ancestors")

print("""
  TAKEAWAY:
    GRetriever = best CLINICAL ACCURACY (Vertigo = correct for R42)
    GNN        = best COVERAGE REACH (broad terms match more diseases)

    They are complementary:
      Use GNN reranking for disease SCREENING (maximize recall)
      Use GRetriever for phenotype ANNOTATION (maximize precision)
""")

=== ICD R42: Dizziness and giddiness ===

  Qwen:       HP:0010532  Paroxysmal vertigo
  GNN:        HP:0002018  Nausea
  GRetriever: HP:0002321  Vertigo

  Clinical assessment:
    Qwen 'Paroxysmal vertigo' -- too specific (implies episodic BPPV)
    GNN  'Nausea' -- wrong symptom (nausea != dizziness)
    GRetriever 'Vertigo' -- correct (dizziness = vertigo)

  Disease coverage impact:
    [A] Qwen         HP:0010532  ->    7 direct diseases,   257 via ancestors
    [B] GNN          HP:0002018  ->  136 direct diseases,   298 via ancestors
    [C] GRetriever   HP:0002321  ->  143 direct diseases,   251 via ancestors

  TAKEAWAY:
    GRetriever = best CLINICAL ACCURACY (Vertigo = correct for R42)
    GNN        = best COVERAGE REACH (broad terms match more diseases)

    They are complementary:
      Use GNN reranking for disease SCREENING (maximize recall)
      Use GRetriever for phenotype ANNOTATION (maximize precision)



In [33]:
# 5-case comparison: GNN precision vs GRetriever precision
showcase_cases = [
    ("R42",  "Dizziness and giddiness"),
    ("F43.8", "Other reactions to severe stress"),
    ("H53.9", "Visual disturbance, unspecified"),
    ("K07.2", "Anomalies of dental arch relationship"),
    ("R41.8", "Other symptoms involving cognitive functions"),
]

print(f"{'ICD':<8} {'Label':<35} {'Qwen':<28} {'GNN':<28} {'GRetriever':<28} {'Best accuracy'}")
print("=" * 155)

for icd_code, icd_desc in showcase_cases:
    if icd_code not in mappings["qwen"]:
        continue
    q = mappings["qwen"][icd_code]
    g = mappings["gnn"][icd_code]
    r = mappings["gretriever"][icd_code]

    best = {"R42": "GRetriever", "F43.8": "GRetriever", "H53.9": "GNN",
            "K07.2": "GNN", "R41.8": "GNN"}.get(icd_code, "?")
    print(f"{icd_code:<8} {icd_desc[:33]:<35} {q['label'][:26]:<28} {g['label'][:26]:<28} {r['label'][:26]:<28} {best}")

print("""
Summary:
  GNN wins on accuracy:        3/5 cases (broader but correct terms)
  GRetriever wins on accuracy: 2/5 cases (precise clinical terms)
  Both methods add value over Qwen-only in ALL 5 cases.
""")

ICD      Label                               Qwen                         GNN                          GRetriever                   Best accuracy
R42      Dizziness and giddiness             Paroxysmal vertigo           Nausea                       Vertigo                      GRetriever
F43.8    Other reactions to severe stress    Intense psychological dist   Panic attack                 Dissociation                 GRetriever
H53.9    Visual disturbance, unspecified     Transient unilateral blurr   Visual impairment            Visual loss                  GNN
K07.2    Anomalies of dental arch relation   Abnormal dental morphology   Dental malocclusion          Abnormality of the dentiti   GNN
R41.8    Other symptoms involving cognitiv   Abnormality of mental func   Cognitive impairment         obsolete Focal impaired aw   GNN

Summary:
  GNN wins on accuracy:        3/5 cases (broader but correct terms)
  GRetriever wins on accuracy: 2/5 cases (precise clinical terms)
  Both methods 